# Inicializando

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType, IntegerType
from pyspark.sql.functions import col, trim, to_date

# Lendo dados da Tabela Bronze

In [0]:
df = spark.table("workspace.bronze.crm_prd_info")
df.display()

# Remoção de espaços desnecessários de colunas string

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))
df.display()

# Conversão de Product Keys

In [0]:
df = df.withColumn("cat_id", F.substring(col("prd_key"), 1, 5))
df = df.withColumn("prd_key", F.substring(col("prd_key"), 7, F.length(col("prd_key"))))

# Normalização da coluna 'prd_line'

In [0]:
df = (
    df.withColumn(
        'prd_line', 
        F.when(F.upper(F.col("prd_line")) == 'M', 'Mountain')
        .when(F.upper(F.col("prd_line")) == 'R', 'Road')
        .when(F.upper(F.col("prd_line")) == 'S', 'Other Sales')
        .when(F.upper(F.col("prd_line")) == 'T', 'Touring')
        .otherwise("n/a")
        )
)
df.display()


# Tratamento de valores nulos na coluna 'prd_cost'

In [0]:
df = (
    df.withColumn('prd_cost',
        F.when(F.isnull(F.col('prd_cost')), 0)
        .otherwise(F.col('prd_cost'))
    )
)

df.display()

# Renomeando Colunas

In [0]:
RENAME_MAP = {
    'cat_id': 'category_id',
    'prd_id': 'product_id',
    'prd_key': 'product_number',
    'prd_nm': 'product_name',
    'prd_cost': 'product_cost',
    'prd_line': 'product_line',
    'prd_start_dt': 'start_date',
    'prd_end_dt': 'end_date'
}

for old_name, new_name in RENAME_MAP.items():
  df = df.withColumnRenamed(old_name, new_name)
  
df.display()

# Criando Tabela Silver

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_products")

# Verificação de Dados na tabela final

In [0]:
%sql

SELECT * FROM workspace.silver.crm_products;